# Metadata aggregation

**Purpose.** Combine lake metadata used by the downstream analysis.

**Inputs.** Local lake metadata and intermediate lookup tables.

**Outputs.** Consolidated metadata tables.

> Historical research notebook. Paths assume the repository layout described in `data/README.md`; generated outputs are intentionally not stored in the notebook.


In [ ]:
from pathlib import Path
import os

start_dir = Path.cwd().resolve()
for candidate in (start_dir, *start_dir.parents):
    if (candidate / 'notebooks').is_dir() and (candidate / 'README.md').is_file():
        os.chdir(candidate)
        break
else:
    raise RuntimeError('Run this notebook from inside the cloned repository.')


In [ ]:
import os
import glob
import pandas as pd

def add_lake_parameters_to_each_file(input_folder, lake_param_df, output_folder):
    os.makedirs(output_folder, exist_ok=True)

    for file in glob.glob(os.path.join(input_folder, "Lake_*.csv")):
        try:
            df = pd.read_csv(file)
            lake_id = int(os.path.basename(file).split("_")[1].split(".")[0])

            # Get lake-level parameters
            lake_meta = lake_param_df[lake_param_df["Lake_ID"] == lake_id]
            if lake_meta.empty:
                print(f"Lake_ID {lake_id} not found in parameters file. Skipping.")
                continue

            # Add parameters to all rows
            for col in lake_meta.columns:
                if col != "Lake_ID":
                    df[col] = lake_meta.iloc[0][col]

            # Save to new output folder
            out_path = os.path.join(output_folder, f"Lake_{lake_id}.csv")
            df.to_csv(out_path, index=False)
            print(f"Saved: {out_path}")

        except Exception as e:
            print(f"Error processing {file}: {e}")



In [ ]:
lake_params = pd.read_excel("Datasets/CNR/metadata.xlsx")
required_columns = [
     "id_int", "Lake_area", "Shore_len", "Shore_dev", "Vol_total",
     "Depth_avg", "Dis_avg", "Res_time", "Elevation", "Slope_100", "Wshd_area"
 ]
lake_params_filtered = lake_params[required_columns].rename(columns={"id_int": "Lake_ID"})

add_lake_parameters_to_each_file("Datasets/Merged_Lake_CSVs", lake_params_filtered, "Datasets/Merged_With_Metadata")
